In [ ]:
import os
import shutil
import zipfile


# paths
judgements_dir_train = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_train"
judgements_dir_test = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_test"

if os.path.exists(judgements_dir_train):
    shutil.rmtree(judgements_dir_train)
    print(f"Deleted: {judgements_dir_train}")
    

if os.path.exists(judgements_dir_test):
    shutil.rmtree(judgements_dir_test)
    print(f"Deleted: {judgements_dir_test}")

In [ ]:
import zipfile

path_to_zip_file = "/u/home/i/iacir21/myscratch/judgements_txt.zip"

with zipfile.ZipFile(path_to_zip_file, 'r') as zip_ref:
    zip_ref.extractall()
    

In [1]:
import os
cleaned_dir= "/u/home/i/iacir21/myscratch/train_test_set/cleaned_text_files"
judgements_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt"

file_count = len([f for f in os.listdir(judgements_dir) if os.path.isfile(os.path.join(judgements_dir, f))])

print(f"Number of files in {judgements_dir}: {file_count}")

file_count = len([f for f in os.listdir(cleaned_dir) if os.path.isfile(os.path.join(cleaned_dir, f))])

print(f"Number of files in {cleaned_dir}: {file_count}")

Number of files in /u/home/i/iacir21/myscratch/train_test_set/judgements_txt: 163130
Number of files in /u/home/i/iacir21/myscratch/train_test_set/cleaned_text_files: 31318


In [ ]:
import shutil

judgements_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt"

cleaned_dir = "/u/home/i/iacir21/myscratch/train_test_set/cleaned_text_files"

# 3. Copy all cleaned files into the new judgements_txt
for file_name in os.listdir(cleaned_dir):
    src_path = os.path.join(cleaned_dir, file_name)
    dst_path = os.path.join(judgements_dir, file_name)

    if os.path.isfile(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"Copied: {file_name}")
        
        
        
file_count = len([f for f in os.listdir(judgements_dir) if os.path.isfile(os.path.join(judgements_dir, f))])

print(f"Number of files in {judgements_dir}: {file_count}")

In [1]:
import os
cleaned_dir= "/u/home/i/iacir21/myscratch/extracted/u/home/i/iacir21/myscratch/train_test_set/cleaned_text_files"
judgements_dir = "/u/home/i/iacir21/myscratch/extracted/u/home/i/iacir21/myscratch/train_test_set/judgements_txt"

file_count = len([f for f in os.listdir(judgements_dir) if os.path.isfile(os.path.join(judgements_dir, f))])

print(f"Number of files in {judgements_dir}: {file_count}")

file_count = len([f for f in os.listdir(cleaned_dir) if os.path.isfile(os.path.join(cleaned_dir, f))])

print(f"Number of files in {cleaned_dir}: {file_count}")

Number of files in /u/home/i/iacir21/myscratch/extracted/u/home/i/iacir21/myscratch/train_test_set/judgements_txt: 163130
Number of files in /u/home/i/iacir21/myscratch/extracted/u/home/i/iacir21/myscratch/train_test_set/cleaned_text_files: 31318


In [2]:


# paths to your folders
cleaned_folder = "/u/home/i/iacir21/myscratch/train_test_set/cleaned_text_files"
judgements_folder = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt"

# get filenames (without paths)
cleaned_files = set(os.listdir(cleaned_folder))
judgement_files = set(os.listdir(judgements_folder))

# find duplicates
duplicates = cleaned_files.intersection(judgement_files)

print(len(duplicates))

31318


In [7]:
import shutil
from pathlib import Path
import pandas as pd

# ---------- CONFIG ----------
MAIN_DIR   = Path("/u/home/i/iacir21/myscratch")
SRC_DIR    = MAIN_DIR / "extracted/u/home/i/iacir21/myscratch/train_test_set/judgements_txt"

BASE_OUT   = MAIN_DIR / "train_test_set"
TRAIN_DIR  = BASE_OUT / "judgements_txt_train_vnorm"
TEST_DIR   = BASE_OUT / "judgements_txt_test_vnorm"
ELIG_DIR   = BASE_OUT / "judgements_txt_eligible_vnorm"

XLSX_PATH  = MAIN_DIR / "regression_replication/IND_KEN_V12.csv"

CASE_COL   = "case_id"      # case id column in the xlsx
NUMJ_COL   = "Num_Judges"    # judge count column
YEAR_COL   = "delivery_year"  # filing year column
JUDGE_COL  = "judge_name_y"       # judge(s) column

# include median year rows in train (True) or use strictly < median (False)
INCLUDE_MEDIAN = True

# overwrite existing files in output dirs?
OVERWRITE = True
# ----------------------------

for d in [TRAIN_DIR, TEST_DIR, ELIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def copy_files(case_ids, dest_dir, available_map, overwrite=True):
    copied, missing, skipped = 0, 0, 0
    for cid in case_ids:
        src = available_map.get(cid)
        if not src or not src.exists():
            missing += 1
            continue

        dest = dest_dir / src.name
        if dest.exists() and not overwrite:
            skipped += 1
            continue

        shutil.copy2(str(src), str(dest))
        copied += 1
    return copied, missing, skipped

# 1) collect available .txt files (case_id -> path), EXACT stem match
assert SRC_DIR.is_dir(), f"Missing source dir: {SRC_DIR}"
txt_files = [p for p in SRC_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]
available = {p.stem: p for p in txt_files}

# 2) read Excel & sanity checks
df = pd.read_csv(XLSX_PATH)
required_cols = [CASE_COL, NUMJ_COL, YEAR_COL, JUDGE_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in {XLSX_PATH}: {missing_cols}. Got: {list(df.columns)}")

# 3) ELIGIBILITY FILTERS (these define the eligible pool)

# 3a) must match an existing txt file (exact)
df[CASE_COL] = df[CASE_COL].astype(str).str.strip()
eligible_df = df[df[CASE_COL].isin(available.keys())].copy()

# 3b) must be single-judge: Num_Judges == 1
eligible_df = eligible_df[eligible_df[NUMJ_COL] == 1].copy()


# 3d) filing_year numeric and judge present
eligible_df[YEAR_COL] = pd.to_numeric(eligible_df[YEAR_COL], errors="coerce")
eligible_df = eligible_df.dropna(subset=[YEAR_COL, JUDGE_COL]).copy()

# 3e) must have a non-null median_slant_goodvbad
eligible_df = eligible_df[eligible_df["median_slant_goodvbad"].notna()].copy()

# 4) per-judge median filing year (computed on eligible pool)
med_year = eligible_df.groupby(JUDGE_COL)[YEAR_COL].median().rename("median_year")
eligible_df = eligible_df.merge(med_year, left_on=JUDGE_COL, right_index=True, how="left")

# 5) split eligible -> train/test using median rule
if INCLUDE_MEDIAN:
    train_df = eligible_df[eligible_df[YEAR_COL] <= eligible_df["median_year"]].copy()
else:
    train_df = eligible_df[eligible_df[YEAR_COL] < eligible_df["median_year"]].copy()

# "Remaining eligible rows" become test
test_df = eligible_df.loc[~eligible_df.index.isin(train_df.index)].copy()

# 6) unique case_id lists
eligible_ids = eligible_df[CASE_COL].drop_duplicates().tolist()
train_ids    = train_df[CASE_COL].drop_duplicates().tolist()
test_ids     = test_df[CASE_COL].drop_duplicates().tolist()

# 7) Copy files
train_copied, train_missing, train_skipped = copy_files(train_ids, TRAIN_DIR, available, overwrite=OVERWRITE)
test_copied,  test_missing,  test_skipped  = copy_files(test_ids,  TEST_DIR,  available, overwrite=OVERWRITE)
elig_copied,  elig_missing,  elig_skipped  = copy_files(eligible_ids, ELIG_DIR, available, overwrite=OVERWRITE)

# 8) Export ID lists
(pd.Series(train_ids, name="case_id")
   .to_csv(BASE_OUT / "train_case_ids_norm.csv", index=False))
(pd.Series(test_ids, name="case_id")
   .to_csv(BASE_OUT / "test_case_ids_vnorm.csv", index=False))
(pd.Series(eligible_ids, name="case_id")
   .to_csv(BASE_OUT / "eligible_case_ids_vnorm.csv", index=False))

# 9) Per-judge summary CSV (eligible + train + test counts)
eligible_counts = eligible_df.groupby(JUDGE_COL).size().rename("eligible_docs")
train_counts    = train_df.groupby(JUDGE_COL).size().rename("train_docs")
test_counts     = test_df.groupby(JUDGE_COL).size().rename("test_docs")

judge_summary = (
    med_year.to_frame()
    .join([eligible_counts, train_counts, test_counts])
    .fillna(0)
    .reset_index()
)
judge_summary.columns = [JUDGE_COL, "median_year", "eligible_docs", "train_docs", "test_docs"]
judge_summary_path = BASE_OUT / "median_year_judges_vnorm.csv"
judge_summary.to_csv(judge_summary_path, index=False)

# 10) Judge -> case_ids reports (train/test)
train_judge_cases = (
    train_df.groupby(JUDGE_COL)[CASE_COL]
    .apply(lambda ids: ";".join(sorted(ids.unique())))
    .reset_index()
    .rename(columns={CASE_COL: "train_case_ids"})
)
train_judge_cases["train_docs"] = train_judge_cases["train_case_ids"].apply(lambda x: 0 if x == "" else len(x.split(";")))
train_judge_cases_path = BASE_OUT / "judge_train_caseids_vnorm.csv"
train_judge_cases.to_csv(train_judge_cases_path, index=False)

test_judge_cases = (
    test_df.groupby(JUDGE_COL)[CASE_COL]
    .apply(lambda ids: ";".join(sorted(ids.unique())))
    .reset_index()
    .rename(columns={CASE_COL: "test_case_ids"})
)
test_judge_cases["test_docs"] = test_judge_cases["test_case_ids"].apply(lambda x: 0 if x == "" else len(x.split(";")))
test_judge_cases_path = BASE_OUT / "judge_test_caseids_vnorm.csv"
test_judge_cases.to_csv(test_judge_cases_path, index=False)

# 11) Summary prints
print("Done.")
print(f"Total .txt in source:                    {len(txt_files)}")

print(f"Eligible rows after all filters:          {len(eligible_df)}")
print(f"Train rows (median rule):                 {len(train_df)}")
print(f"Test rows (remaining eligible):           {len(test_df)}")

print(f"Unique eligible case_ids:                 {len(eligible_ids)}")
print(f"Unique train case_ids:                    {len(train_ids)}")
print(f"Unique test case_ids:                     {len(test_ids)}")

print("--- Copy results ---")
print(f"TRAIN copied/missing/skipped:             {train_copied}/{train_missing}/{train_skipped}")
print(f"TEST  copied/missing/skipped:             {test_copied}/{test_missing}/{test_skipped}")
print(f"ELIG  copied/missing/skipped:             {elig_copied}/{elig_missing}/{elig_skipped}")

print("--- Outputs ---")
print(f"Train dir:                                {TRAIN_DIR}")
print(f"Test dir:                                 {TEST_DIR}")
print(f"Eligible dir:                             {ELIG_DIR}")
print(f"Train IDs CSV:                            {BASE_OUT / 'train_case_ids_vnorm.csv'}")
print(f"Test IDs CSV:                             {BASE_OUT / 'test_case_ids_vnorm.csv'}")
print(f"Eligible IDs CSV:                         {BASE_OUT / 'eligible_case_ids_vnorm.csv'}")
print(f"Judge median/counts CSV:                  {judge_summary_path}")
print(f"Judge train case_ids CSV:                 {train_judge_cases_path}")
print(f"Judge test case_ids CSV:                  {test_judge_cases_path}")
print(f"Inclusion rule: filing_year {'<= median' if INCLUDE_MEDIAN else '< median'}")

Done.
Total .txt in source:                    163129
Eligible rows after all filters:          26957
Train rows (median rule):                 16193
Test rows (remaining eligible):           10764
Unique eligible case_ids:                 26957
Unique train case_ids:                    16193
Unique test case_ids:                     10764
--- Copy results ---
TRAIN copied/missing/skipped:             16193/0/0
TEST  copied/missing/skipped:             10764/0/0
ELIG  copied/missing/skipped:             26957/0/0
--- Outputs ---
Train dir:                                /u/home/i/iacir21/myscratch/train_test_set/judgements_txt_train_vnorm
Test dir:                                 /u/home/i/iacir21/myscratch/train_test_set/judgements_txt_test_vnorm
Eligible dir:                             /u/home/i/iacir21/myscratch/train_test_set/judgements_txt_eligible_vnorm
Train IDs CSV:                            /u/home/i/iacir21/myscratch/train_test_set/train_case_ids_vnorm.csv
Test IDs CSV:    

In [4]:
import shutil
from pathlib import Path
import pandas as pd

# ---------- CONFIG ----------
MAIN_DIR   = Path("/u/home/i/iacir21/myscratch")
SRC_DIR    = MAIN_DIR / "train_test_set/judgements_txt"

BASE_OUT   = MAIN_DIR / "train_test_set"
TRAIN_DIR  = BASE_OUT / "judgements_txt_train_vnorm"
TEST_DIR   = BASE_OUT / "judgements_txt_test_vnorm"
ELIG_DIR   = BASE_OUT / "judgements_txt_eligible_vnorm"

XLSX_PATH  = MAIN_DIR / "regression_replication/IND_KEN_V12.csv"

CASE_COL   = "case_id"      # case id column in the xlsx
NUMJ_COL   = "Num_Judges"    # judge count column
YEAR_COL   = "delivery_year"  # filing year column
JUDGE_COL  = "judge_name_y"       # judge(s) column

df = pd.read_csv(XLSX_PATH)

mask = (
    df["median_slant_goodvbad"].notna() &
    pd.to_numeric(df[YEAR_COL], errors="coerce").isna()
)

print(f"Rows where median_slant_goodvbad is not null AND {YEAR_COL} is non-numeric: {mask.sum()}")

Rows where median_slant_goodvbad is not null AND delivery_year is non-numeric: 0
